# Connect to Serverless compute to run this notebook
To begin, connect to the Serverless compute resource that will execute this batch inference notebook.

In [0]:
# Install MLflow
%pip install mlflow
dbutils.library.restartPython()

# Load Model from Unity Catalog and Install Dependencies
In this step, we load the registered model from Unity Catalog and install any dependencies required for inference.

In [0]:
import mlflow.pyfunc
import os

os.environ['MLFLOW_USE_DATABRICKS_SDK_MODEL_ARTIFACTS_REPO_FOR_UC'] = 'True'

uc_model_path = "workspace.default.sudenergy_lux_temp_forecast"
model_version_uri = f"models:/{uc_model_path}@most_recent"
mlflow.set_registry_uri("databricks-uc")
requirements = mlflow.pyfunc.get_model_dependencies(model_version_uri)
%pip install -r {requirements}
dbutils.library.restartPython()

# Prepare Inference Table
We now prepare the inference table. The following parameters define the table configuration.
- **time_column**: The column representing time in your dataset.
- **target_column**: The column representing the prediction target in your dataset.
- **frequency_unit**: The frequency of the forecast intervals. See https://pandas.pydata.org/docs/reference/api/pandas.date_range.html for supported frequency string.
- **frequency_quantity**: The frequency quantity of the frequency unit.
- **forecast_horizon**: The number of time intervals to forecast.

In [0]:
import pyspark.sql.functions as F
import pandas as pd

# Define parameters


train_data_path = "workspace.default.silver_weather_daily"
train_data_path = '.'.join([f'`{part}`' for part in train_data_path.split('.')]) # escape special characters for unity catalog path
time_column = "date"
target_column = "avg_temp"
frequency_unit = "D"
frequency_quantity = 1
forecast_horizon = 365


history_df = spark.table(train_data_path)
history_df = history_df.withColumn(time_column, F.to_timestamp(time_column))


# You can change the start time to fit your use case, here start time is set to be the last timestamp in the training dataset, history_df
start_time = history_df.select(F.max(time_column)).collect()[0][0]



# Generate the date range starting from start_time
date_range = pd.date_range(start=pd.to_datetime(start_time), periods=forecast_horizon+1, freq=f"{frequency_quantity}{frequency_unit}")[1:]


data = {
  time_column: list(date_range),
    }

predict_df = pd.DataFrame(data)
display(predict_df)

# Load the Model and Execute Batch Inference
Using the model, we now perform batch inference on the generated date range.

In [0]:
import mlflow
import os
os.environ['MLFLOW_USE_DATABRICKS_SDK_MODEL_ARTIFACTS_REPO_FOR_UC'] = 'True'

uc_model_path = "workspace.default.sudenergy_lux_temp_forecast"
model_version_uri = f"models:/{uc_model_path}@most_recent"
mlflow.set_registry_uri("databricks-uc")
model = mlflow.pyfunc.load_model(model_version_uri)
results_df = model.predict(predict_df)

# Display the prediction results with timestamp
final_df = pd.concat([predict_df, results_df], axis=1)
final_df = final_df.rename(columns={'yhat': target_column})
display(final_df)

# Plot the prediction results
Visualize the predictions made by the model alongside recent historical data. The historical data is truncated to maintain clarity and conciseness in the plot.

In [0]:
import pandas as pd
import matplotlib.pyplot as plt


try:
  num_history_horizons = 9
  # Get (up to) the last (num_history_horizons * forecast_horizon) rows from the training dataset, history_df.
  history_df = history_df.orderBy(time_column, ascending=False).limit(num_history_horizons * forecast_horizon).toPandas()

  # Code for plotting
  plt.figure(figsize=(10, 6))

  # Plot the solid line for the historical datapoints.
  plt.plot(history_df[time_column], history_df[target_column], linestyle='-', marker='s', label='Historical values')

  # Plot the dashed line for the forecasted values
  plt.plot(final_df[time_column], final_df[target_column], linestyle='--', marker='o', label='Forecasts')

  # Set proper ticks on x-axis.
  num_ticks = num_history_horizons + 1
  tick_positions = pd.date_range(start=history_df[time_column].min(), end=final_df[time_column].max(), periods=num_ticks)
  plt.xticks(tick_positions, labels=[date.strftime("%Y-%m-%d %H:%M") for date in tick_positions], rotation=45)

  # Adding labels and legend
  plt.title('Recent Historical Data and Forecasts')
  plt.xlabel(time_column)
  plt.ylabel(target_column)
  plt.legend()

  # Display the plot
  plt.show()

except Exception as e:
    print(f"Failed to generate plot due to an exception: {e}")